# Inverse Reinforcement Learning

Wiki reference for [inverse reinforcement learning](https://ml-viz-ruby.vercel.app/wiki/inverse-reinforcement-learning).

**The idea in one sentence.** IRL recovers the **reward** an expert is optimizing from its
demonstrations — but the problem is fundamentally **ill-posed** (many rewards explain the same
behavior: a constant shift or positive rescaling leaves the optimal policy unchanged), so
**MaxEnt IRL** picks the reward whose soft-optimal policy **matches the expert's state-visitation
frequencies**.

We build a gridworld, value iteration, and the MaxEnt forward pass from scratch, **validate the
soft-optimal policy and its visitation, plus the reward ambiguity**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748', 'grid.color': '#2d3748', 'axes.grid': False,
})
np.random.seed(0)

## 1. Grid world + true (hidden) reward

In [ ]:
N = 5  # 5×5 grid
n_states = N * N
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]  # up, down, left, right
n_actions = len(ACTIONS)
GAMMA = 0.9

# True reward: goal at corner (4,4), small penalty elsewhere
true_reward = -np.ones(n_states) * 0.1
GOAL = 4 * N + 4
true_reward[GOAL] = 1.0

def s_to_rc(s): return divmod(s, N)
def rc_to_s(r, c): return r * N + c

# Transition (deterministic, stay in bounds)
P = np.zeros((n_states, n_actions, n_states))
for s in range(n_states):
    r, c = s_to_rc(s)
    for a, (dr, dc) in enumerate(ACTIONS):
        nr, nc = np.clip(r+dr, 0, N-1), np.clip(c+dc, 0, N-1)
        P[s, a, rc_to_s(nr, nc)] = 1.0

def value_iteration(reward, n_iters=100):
    V = np.zeros(n_states)
    for _ in range(n_iters):
        Q = reward[:, None] + GAMMA * (P @ V)  # (s, a)
        V = Q.max(axis=1)
    Q = reward[:, None] + GAMMA * (P @ V)
    return V, Q

V_true, Q_true = value_iteration(true_reward)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(true_reward.reshape(N, N), cmap='RdYlGn')
ax.set_title('True (hidden) reward', color='#e2e8f0')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 2. Generate expert demonstrations

The expert follows a softmax (Boltzmann) policy over the true Q-values — approximately optimal, as MaxEnt IRL assumes.

In [ ]:
def softmax_policy(Q, beta=5.0):
    e = np.exp(beta * (Q - Q.max(axis=1, keepdims=True)))
    return e / e.sum(axis=1, keepdims=True)

expert_policy = softmax_policy(Q_true)

def sample_trajectory(policy, start, length=12):
    s = start
    traj = [s]
    for _ in range(length):
        a = np.random.choice(n_actions, p=policy[s])
        s = np.random.choice(n_states, p=P[s, a])
        traj.append(s)
    return traj

n_demos = 200
demos = [sample_trajectory(expert_policy, start=np.random.randint(n_states)) for _ in range(n_demos)]

# Expert state visitation frequency (feature expectation)
expert_svf = np.zeros(n_states)
for traj in demos:
    for s in traj:
        expert_svf[s] += 1
expert_svf /= expert_svf.sum()

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(expert_svf.reshape(N, N), cmap='magma')
ax.set_title('Expert state-visitation frequency', color='#e2e8f0')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### Validate: the expert follows the soft-optimal policy

The expert acts by a softmax over the true $Q$-values, so it puts the **most** probability on
the highest-value action in each state (with some exploration). We confirm the policy's mode
matches the greedy action everywhere.

In [ ]:
matches = all(expert_policy[s].argmax() == Q_true[s].argmax() for s in range(n_states))
print(f'expert softmax mode == greedy action in every state: {matches}')
assert matches, 'the softmax expert favors the highest-value action in each state'
print('\n✅ the expert is a (soft) optimizer of the true reward — the behavior IRL sees')

## 3. MaxEnt IRL — recover the reward

We use one-hot state features, so the recovered reward is a per-state weight vector. The gradient matches expert state-visitation to the learner's expected visitation under the current reward.

In [ ]:
def expected_svf(reward, n_steps=30):
    """Forward pass: expected state-visitation under the soft-optimal policy for `reward`."""
    _, Q = value_iteration(reward, n_iters=80)
    policy = softmax_policy(Q)
    # Initial distribution = expert start distribution (uniform here)
    mu = np.ones(n_states) / n_states
    svf = mu.copy()
    for _ in range(n_steps):
        mu_next = np.zeros(n_states)
        for s in range(n_states):
            for a in range(n_actions):
                mu_next += mu[s] * policy[s, a] * P[s, a]
        mu = mu_next
        svf += mu
    return svf / svf.sum()

# Gradient ascent on reward weights
reward_est = np.zeros(n_states)
lr = 0.5
history = []
for it in range(60):
    learner_svf = expected_svf(reward_est)
    grad = expert_svf - learner_svf   # match feature expectations
    reward_est += lr * grad
    history.append(np.corrcoef(reward_est, true_reward)[0, 1])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im0 = axes[0].imshow(true_reward.reshape(N, N), cmap='RdYlGn')
axes[0].set_title('True reward', color='#e2e8f0'); plt.colorbar(im0, ax=axes[0], fraction=0.046)
# Normalize recovered reward for comparison
r_norm = (reward_est - reward_est.mean()) / reward_est.std()
im1 = axes[1].imshow(r_norm.reshape(N, N), cmap='RdYlGn')
axes[1].set_title('Recovered reward (MaxEnt IRL)', color='#e2e8f0'); plt.colorbar(im1, ax=axes[1], fraction=0.046)
axes[2].plot(history, color='#6366f1', linewidth=2)
axes[2].set_xlabel('IRL iteration'); axes[2].set_ylabel('Corr(recovered, true)')
axes[2].set_title('Reward Recovery Correlation', color='#e2e8f0'); axes[2].grid(True)
plt.tight_layout()
plt.show()

print(f"Final correlation with true reward: {history[-1]:.3f}")
print(f"Recovered argmax state: {reward_est.argmax()} (true goal: {GOAL})")

### Validate: state-visitation concentrates at the goal

MaxEnt IRL's forward pass computes the **expected state-visitation frequency** (SVF) of the
soft-optimal policy — the quantity it matches to the expert's. Under the true reward the SVF is
far higher at the goal than average. We confirm.

In [ ]:
svf = expected_svf(true_reward)
print(f'SVF at goal = {svf[GOAL]:.3f}  vs mean SVF = {svf.mean():.3f}')
assert svf[GOAL] > svf.mean(), 'the soft-optimal policy visits the goal far more than an average state'
print('\n✅ MaxEnt IRL matches these visitation frequencies to recover the reward')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **reward ambiguity** | shift & scale don't change behavior (demo) — reward is under-determined |
| **needs a solver in the loop** | IRL repeatedly solves the forward RL problem — expensive |
| **suboptimal demos** | noisy experts bias the recovered reward |
| **feature choice** | linear-reward IRL is only as expressive as its features |
| **reward vs policy** | if you only need the policy, imitation may be simpler than IRL |

Demo: constant shift and positive scaling leave the optimal policy unchanged.

In [ ]:
# Why IRL is HARD: the reward is under-determined. The optimal policy is invariant to (a) adding
# a constant to the reward and (b) multiplying by a positive scalar — so infinitely many rewards
# produce the SAME behavior. From demonstrations alone you cannot recover the reward's offset or
# scale. We confirm both invariances.
_, Q_base = value_iteration(true_reward)
_, Q_shift = value_iteration(true_reward + 10.0)   # add a constant
_, Q_scale = value_iteration(true_reward * 3.0)    # positive scaling
pol = lambda Q: Q.argmax(axis=1)
print('policy unchanged by +constant:', np.array_equal(pol(Q_base), pol(Q_shift)))
print('policy unchanged by x positive:', np.array_equal(pol(Q_base), pol(Q_scale)))
assert np.array_equal(pol(Q_base), pol(Q_shift)), 'adding a constant leaves the optimal policy unchanged'
assert np.array_equal(pol(Q_base), pol(Q_scale)), 'positive scaling leaves the optimal policy unchanged'
print('\nMany rewards -> the same behavior. IRL is ill-posed -> MaxEnt picks one by matching visitation frequencies.')

## ✏️ Your turn

**Exercise 1 — Reward ambiguity.** Add a constant to the recovered reward and re-run value iteration. Verify the optimal policy is unchanged — demonstrating that IRL can only recover the reward up to constant shift (and scaling).

In [ ]:
# TODO(you): compute the greedy policy from Q_true and from value_iteration(true_reward + 10.0)
# then check they are identical

In [ ]:
# Assert cell
_, Q_a = value_iteration(true_reward)
_, Q_b = value_iteration(true_reward + 10.0)   # shifted reward
policy_a = Q_a.argmax(axis=1)
policy_b = Q_b.argmax(axis=1)
print("Policies identical after constant shift:", np.array_equal(policy_a, policy_b))
assert np.array_equal(policy_a, policy_b), "Adding a constant must not change the optimal policy"

<details><summary>Solution</summary>

```python
_, Q_a = value_iteration(true_reward)
_, Q_b = value_iteration(true_reward + 10.0)
assert np.array_equal(Q_a.argmax(1), Q_b.argmax(1))
```

Adding a constant `c` to every state's reward adds `c/(1-γ)` to every state's value uniformly, so the *relative* ordering of actions — and hence the optimal policy — is unchanged. This is exactly the reward-ambiguity problem: infinitely many reward functions explain the same behavior, which is why MaxEnt IRL needs the maximum-entropy tie-breaker.

</details>

## Key takeaways

- **IRL recovers the reward** behind expert demonstrations.
- **Soft-optimal expert:** the softmax policy favors high-value actions (verified).
- **Visitation matching:** MaxEnt IRL matches the expert's state-visitation frequencies
  (verified).
- **Ill-posed:** constant shift and positive scaling leave behavior unchanged (demo) — MaxEnt
  resolves the ambiguity with a maximum-entropy principle.